# Toy DINO vs Graha Instance-Label Comparison

This notebook mirrors `toy_inst_seg_comparison.py` for interactive runs. It uses instance `.npz` labels as binary segmentation targets: the Toy DINO path trains on `mask > 0`, while the Graha path uses the instance datamodule with binary masks and keeps crater boxes available for shape loss.

## Imports and Paths

In [ ]:
from argparse import Namespace
from pathlib import Path
import gc
import json
import sys
import time

import torch
from lightning.pytorch import seed_everything

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "notebooks" / "full_model", cwd.parent / "notebooks" / "full_model"]
NOTEBOOK_DIR = next(path for path in candidates if (path / "toy_inst_seg_comparison.py").exists())
REPO_ROOT = NOTEBOOK_DIR.parents[1]
for path in [REPO_ROOT, NOTEBOOK_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from lfm.full_model.utils.utils import ensure_data_symlink
from lfm.full_model.utils import (
    create_timestamped_output_dir,
    evaluate_prediction_caches,
    plot_prediction_cache_comparison,
    save_prediction_cache,
)
from toy_inst_seg_comparison import (
    build_config,
    create_datamodule,
    create_lightning_module,
    create_model,
    create_trainer,
    load_lightning_checkpoint_state,
    record_timing,
    run_graha_workflow,
    save_config,
    save_timing_summary,
    validate_data_paths,
)

NOTEBOOK_DIR, REPO_ROOT

## Configuration

In [ ]:
SIMLINK_DEST = None
DATA_ROOT = None  # None uses NOTEBOOK_DIR / "data".
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "toy_inst_seg_comparison"

DINO_CHECKPOINT = None
DINO_LIGHTNING_CHECKPOINT = None
GRAHA_PRETRAIN_DIR = None
GRAHA_LIGHTNING_CHECKPOINT = None

BAND_FILTER = [0, 1, 2, 3, 4, 5, 6]
TARGET_SIZE = 256
SPATIAL_TRANSFORM = "crop"
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

BATCH_SIZE = 16
NUM_WORKERS = 10
MAX_EPOCHS = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-3
LOSS_TYPE = "focal_dice"
FREEZE_ENCODER = False
NORMALIZE_INPUTS = False

LABEL_FILE_TYPE = ".npz"
LABEL_NPZ_KEY = "mask"
BINARIZE_LABEL = True

TOY_GRADIENT_CLIP_VAL = 1.0
DISABLE_TOY_GRADIENT_CLIPPING = False
PLOT_EVERY_N_EPOCHS = 1
PLOT_N_SAMPLES = 5
CACHE_PREDICTIONS = True
PREDICTION_SPLIT = "val"
PREDICTION_N_SAMPLES = 20

GRAHA_BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "graha_finetuning"
GRAHA_STATS_BATCH_SIZE = 16
GRAHA_BATCH_SIZE = 16
GRAHA_NUM_WORKERS = 10

SKIP_DINO_FIT = False
SKIP_GRAHA_FIT = False
SEED = 42

## Build Shared Run State

In [ ]:
ensure_data_symlink(SIMLINK_DEST, NOTEBOOK_DIR / "data")

args = Namespace(
    data_root=str(DATA_ROOT) if DATA_ROOT is not None else None,
    base_output_dir=str(BASE_OUTPUT_DIR),
    dino_checkpoint=str(DINO_CHECKPOINT) if DINO_CHECKPOINT is not None else None,
    dino_lightning_checkpoint=str(DINO_LIGHTNING_CHECKPOINT) if DINO_LIGHTNING_CHECKPOINT is not None else None,
    band_filter=BAND_FILTER,
    target_size=TARGET_SIZE,
    spatial_transform=SPATIAL_TRANSFORM,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_val_samples=MAX_VAL_SAMPLES,
    max_test_samples=MAX_TEST_SAMPLES,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_epochs=MAX_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    loss_type=LOSS_TYPE,
    freeze_encoder=FREEZE_ENCODER,
    normalize_inputs=NORMALIZE_INPUTS,
    label_file_type=LABEL_FILE_TYPE,
    label_npz_key=LABEL_NPZ_KEY,
    binarize_label=BINARIZE_LABEL,
    toy_gradient_clip_val=TOY_GRADIENT_CLIP_VAL,
    disable_toy_gradient_clipping=DISABLE_TOY_GRADIENT_CLIPPING,
    plot_every_n_epochs=PLOT_EVERY_N_EPOCHS,
    plot_n_samples=PLOT_N_SAMPLES,
    cache_predictions=CACHE_PREDICTIONS,
    prediction_split=PREDICTION_SPLIT,
    prediction_n_samples=PREDICTION_N_SAMPLES,
    graha_base_output_dir=str(GRAHA_BASE_OUTPUT_DIR),
    graha_pretrain_dir=str(GRAHA_PRETRAIN_DIR) if GRAHA_PRETRAIN_DIR is not None else None,
    graha_lightning_checkpoint=str(GRAHA_LIGHTNING_CHECKPOINT) if GRAHA_LIGHTNING_CHECKPOINT is not None else None,
    graha_stats_batch_size=GRAHA_STATS_BATCH_SIZE,
    graha_batch_size=GRAHA_BATCH_SIZE,
    graha_num_workers=GRAHA_NUM_WORKERS,
    no_fit=False,
    skip_dino_fit=SKIP_DINO_FIT,
    skip_graha_fit=SKIP_GRAHA_FIT,
    seed=SEED,
)

config = build_config(args)
validate_data_paths(config)
output_dir = create_timestamped_output_dir(config.base_output_dir)
save_config(config, output_dir)
timing_rows = []
seed_everything(config.seed)
print(f"Using output directory: {output_dir}")

## Toy DINO Datamodule, Model, and Trainer

In [ ]:
dino_total_started_at = time.perf_counter()
toy_datamodule = create_datamodule(config, output_dir)
if toy_datamodule.weight_assignments is None:
    raise RuntimeError("DataModule did not create weight assignments.")

toy_model = create_model(config, toy_datamodule.weight_assignments)
toy_task = create_lightning_module(config, toy_model)
toy_trainer = create_trainer(config, output_dir, plots_subdir=Path("plots") / "toy_model")
print("Toy DINO trainer created.")

## Train or Load Toy DINO

In [ ]:
if config.skip_dino_fit:
    print("Skipping DINO trainer.fit().")
    if config.dino_lightning_checkpoint is not None:
        load_lightning_checkpoint_state(toy_task, config.dino_lightning_checkpoint, "DINO")
else:
    print("Starting DINO trainer.fit()...")
    fit_started_at = time.perf_counter()
    ckpt_path = str(config.dino_lightning_checkpoint) if config.dino_lightning_checkpoint is not None else None
    if ckpt_path is not None:
        print(f"Resuming DINO trainer.fit() from {ckpt_path}")
    toy_trainer.fit(toy_task, datamodule=toy_datamodule, ckpt_path=ckpt_path)
    record_timing(timing_rows, model="DINO", stage="fit", started_at=fit_started_at)
    print("DINO trainer.fit() complete.")

## Cache and Test Toy DINO Predictions

In [ ]:
toy_prediction_cache = None
if config.cache_predictions:
    cache_started_at = time.perf_counter()
    toy_prediction_cache = save_prediction_cache(
        task=toy_task,
        datamodule=toy_datamodule,
        output_dir=output_dir,
        model_name="toy",
        split=config.prediction_split,
        n_samples=config.prediction_n_samples,
    )
    record_timing(timing_rows, model="DINO", stage="prediction_cache", started_at=cache_started_at)

if not config.skip_dino_fit:
    print("Starting DINO trainer.test() on final weights...")
    test_started_at = time.perf_counter()
    toy_trainer.test(toy_task, datamodule=toy_datamodule, ckpt_path=None)
    record_timing(timing_rows, model="DINO", stage="test_final", started_at=test_started_at)
    print("DINO trainer.test() complete.")

del toy_trainer, toy_task, toy_model, toy_datamodule
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released DINO model objects and cleared CUDA cache.")
record_timing(timing_rows, model="DINO", stage="total", started_at=dino_total_started_at)

## Train or Load Graha

In [ ]:
_, graha_prediction_cache = run_graha_workflow(
    config,
    no_fit=config.skip_graha_fit,
    comparison_output_dir=output_dir,
    timing_rows=timing_rows,
)

## Compare Toy DINO and Graha Prediction Caches

In [ ]:
if config.cache_predictions and toy_prediction_cache and graha_prediction_cache:
    comparison_started_at = time.perf_counter()
    comparison_caches = {"toy": toy_prediction_cache, "graha": graha_prediction_cache}
    plot_prediction_cache_comparison(
        comparison_caches,
        output_dir / "comparison_plots",
        n_samples=min(5, config.prediction_n_samples),
    )
    _, metric_summary = evaluate_prediction_caches(
        comparison_caches,
        output_dir / "comparison_metrics",
    )
    print("Comparison metric summary:")
    for row in metric_summary:
        print("  " + json.dumps(row, sort_keys=True))
    record_timing(timing_rows, model="Comparison", stage="plots_and_metrics", started_at=comparison_started_at)

save_timing_summary(timing_rows, output_dir)
print(f"Saved timing summary under {output_dir}")